# 02 — Entrenamiento y Selección del Mejor Modelo

**Proyecto:** Detección de jugadores para offside en fútbol  
**Dataset:** Roboflow `team-separation` v5 — 1.200 imágenes 640×640, 7 clases, ~21.600 bboxes  
**Notebook anterior:** `01_dataset_preparation.ipynb`  
**Hardware asumido:** Google Colab T4 GPU (16 GB VRAM)  

## Objetivo

Fine-tunear un detector de objetos YOLO sobre el dataset de broadcast de fútbol para localizar jugadores (TEAM 1, TEAM 2), árbitros, arqueros, pelota, arco y corners — información necesaria para trazar la línea de offside.

## Reproducibilidad

Se fija `seed=42` en todos los entrenamientos. Con `deterministic=True` se desactivan operaciones no deterministas de CUDA. **Nota:** Ultralytics activa mosaic augmentation que tiene componentes estocásticos aun con semilla fija — esto es esperado y está documentado.

## Experimentos

| Exp | Modelo | Estrategia | Dimensión |
|-----|--------|-----------|----------|
| 1 | YOLOv8s | Full fine-tune | Baseline |
| 2 | YOLOv8s | Freeze backbone (`freeze=10`) | Estrategia de fine-tuning |
| 3 | YOLOv8m | Full fine-tune + focal loss (`fl_gamma=1.5`) | Capacidad + class imbalance |

---
## 1. Configuración del entorno

In [ ]:
# Instalar dependencias (silent en Colab, no-op si ya está instalado)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0", "roboflow>=1.1.0"])

In [ ]:
import os, sys, shutil, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import yaml
import cv2
from PIL import Image
from IPython.display import display

import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
import ultralytics; print(f"Ultralytics: {ultralytics.__version__}")

In [ ]:
# Verificar GPU
if torch.cuda.is_available():
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("ADVERTENCIA: GPU no disponible. El entrenamiento será muy lento en CPU.")
    print("Usar Google Colab con T4 GPU: Runtime > Change runtime type > GPU")

In [ ]:
# Reproducibilidad
# Reutilizamos la misma lógica que src/pipeline.py:48
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Semilla fijada: {SEED}")

---
## 2. Acceso al dataset y configuración de rutas

> **Colab:** Clonar el repositorio y descargar el dataset con el script provisto. El dataset se guarda en `data/raw/` con la estructura YOLO (imágenes + labels + `data.yaml`).

In [ ]:
# ─── Detectar entorno ───────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_ROOT = Path("/content/ProyectoFinalRedesOffside")
    if not REPO_ROOT.exists():
        print("Clonando repositorio...")
        subprocess.run(["git", "clone", "https://github.com/Juampicosta123/ProyectoFinalRedesOffside.git",
                        str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT))
    print(f"Directorio de trabajo: {Path.cwd()}")
else:
    # Local: subir dos niveles desde dev/
    REPO_ROOT = Path("__file__").resolve().parent.parent
    # Si el notebook corre desde dev/, ajustar:
    if Path.cwd().name == "dev":
        os.chdir(Path.cwd().parent)
    REPO_ROOT = Path.cwd()
    print(f"Directorio de trabajo: {REPO_ROOT}")

In [ ]:
# Descargar dataset si no existe
RAW_DIR = Path("data/raw")
IMAGES_TRAIN = RAW_DIR / "train" / "images"

if not IMAGES_TRAIN.exists() or not any(IMAGES_TRAIN.glob("*.jpg")):
    print("Descargando dataset desde Roboflow...")
    subprocess.run([sys.executable, "data/download_dataset.py"], check=True)
else:
    print(f"Dataset ya presente: {sum(1 for _ in IMAGES_TRAIN.glob('*.jpg'))} imágenes en train/images")

In [ ]:
# Configurar data.yaml con ruta absoluta para Ultralytics
# Ultralytics resuelve rutas relativas desde la ubicación del yaml;
# en Colab puede fallar si el CWD no coincide — forzamos ruta absoluta.
YAML_PATH = RAW_DIR / "data.yaml"
assert YAML_PATH.exists(), f"No encontrado: {YAML_PATH}"

with open(YAML_PATH) as f:
    yaml_data = yaml.safe_load(f)

# Reescribir con ruta absoluta del raw dir
abs_raw = str(RAW_DIR.resolve())
yaml_data["path"] = abs_raw
yaml_data["train"] = "train/images"
yaml_data["val"]   = "valid/images"
yaml_data["test"]  = "test/images"

with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_data, f, allow_unicode=True)

DATA_YAML = str(YAML_PATH.resolve())
CLASS_NAMES = yaml_data["names"]
print(f"data.yaml configurado → {DATA_YAML}")
print(f"Clases ({len(CLASS_NAMES)}): {CLASS_NAMES}")

In [ ]:
# Análisis de class imbalance desde los CSVs de splits
train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

counts_train = train_df["class_name"].value_counts().rename("train")
counts_val   = val_df["class_name"].value_counts().rename("val")
counts_test  = test_df["class_name"].value_counts().rename("test")
imbalance_df = pd.concat([counts_train, counts_val, counts_test], axis=1).fillna(0).astype(int)
imbalance_df["total"] = imbalance_df.sum(axis=1)
imbalance_df = imbalance_df.sort_values("total", ascending=False)

print("\n=== Distribución de anotaciones por clase ===")
display(imbalance_df)

fig, ax = plt.subplots(figsize=(10, 4))
imbalance_df["train"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Instancias por clase (train)")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de bboxes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print("\nClases con menor representación → posible impacto en mAP por clase.")

---
## 3. Justificación del modelo preentrenado

### ¿Por qué YOLOv8?

YOLOv8 (Ultralytics, 2023) es un detector single-stage anchor-free que ofrece:
- **Pesos preentrenados en COCO** — COCO incluye clases `person` (≈ jugadores/árbitros) y `sports ball` (≈ pelota), lo que hace el dominio de origen altamente similar al nuestro. Las features del backbone son directamente útiles.
- **Desacople cabeza de detección** — permite reemplazar/fine-tunear la head sin tocar el backbone, clave para la estrategia de freeze.
- **Integración Ultralytics** — métricas automáticas (mAP@50, mAP@50-95, P, R, confusion matrix), CLI simple, reproducible con `seed=42`.

### ¿Por qué YOLOv8s como baseline?

El dataset es pequeño (1.200 imágenes). Un modelo mediano (`s`) equilibra capacidad y riesgo de overfitting. Luego escalamos a `m` para testear ganancia de capacidad.

### Augmentations: Ultralytics vs Albumentations

En el notebook `01_dataset_preparation.ipynb` se definió un pipeline Albumentations (`src/dataset.py:73`). **En Phase 3 ese pipeline NO se usa** — Ultralytics maneja sus propias augmentations internamente (mosaic, mixup, flip, HSV, etc.) que son compatibles con detección de objetos y se aplican correctamente solo a train.  
El parámetro `hsv_h=0.015` (default Ultralytics) es deliberadamente estrecho: preserva los colores de camisetas de TEAM 1 y TEAM 2, que son la señal principal para distinguirlos. Esto es consistente con la justificación del notebook anterior.

---
## 4. Experimento 1 — YOLOv8s Full Fine-Tune (Baseline)

**Configuración:**
- Modelo: `yolov8s.pt` (COCO pretrained)
- Estrategia: full fine-tune (todos los parámetros actualizables)
- Optimizador: AdamW, lr=1e-3, cosine scheduler
- 50 épocas, early stopping patience=10
- batch=16 (T4 16 GB VRAM)

**Justificación de AdamW + cosine:** AdamW es el estándar moderno para transformers y CNNs (L2 regularization desacoplada). Cosine scheduler reduce LR suavemente → mejor convergencia en fine-tuning que step decay.

In [ ]:
set_seed(SEED)
model1 = YOLO("yolov8s.pt")

t0 = time.time()
results1 = model1.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp1_yolov8s_full",
    exist_ok=False,
    verbose=False,
)
t1 = time.time()
print(f"\nExp 1 completado en {(t1 - t0) / 60:.1f} minutos")

del model1
torch.cuda.empty_cache()

---
## 5. Experimento 2 — YOLOv8s Freeze Backbone

**Configuración:** Idéntica al Exp 1, salvo `freeze=10`.

**¿Qué se congela?** `freeze=10` bloquea las primeras 10 capas de YOLOv8s, que corresponden al backbone CSPDarknet. Solo el neck (PANet) y la head se entrenan.

**Justificación de la estrategia:** Con datasets chicos (1.200 imágenes) existe riesgo de sobreescribir features generales del backbone que ya son útiles (COCO incluye personas = jugadores). Congelar el backbone preserva esas features y reduce el espacio de parámetros a optimizar → menos overfitting esperado. La hipótesis es que el neck/head son suficientes para adaptar las features COCO al dominio broadcast de fútbol.

In [ ]:
set_seed(SEED)
model2 = YOLO("yolov8s.pt")

t0 = time.time()
results2 = model2.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    freeze=10,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp2_yolov8s_frozen",
    exist_ok=False,
    verbose=False,
)
t1 = time.time()
print(f"\nExp 2 completado en {(t1 - t0) / 60:.1f} minutos")

del model2
torch.cuda.empty_cache()

---
## 6. Experimento 3 — YOLOv8m Full Fine-Tune + Focal Loss

**Configuración:**
- Modelo: `yolov8m.pt` (mayor capacidad — backbone C2f más profundo)
- `fl_gamma=1.5` — focal loss para class imbalance
- `batch=8` para ajustar VRAM con el modelo más grande

**¿Por qué focal loss?**  
El dataset muestra fuerte desbalance (TEAM 1/TEAM 2 dominan; Goal_Net y Corner son clases raras). `fl_gamma > 0` aplica focal loss sobre la componente de clasificación: los ejemplos fáciles (bien clasificados) reciben menor gradiente, forzando al modelo a enfocarse en los difíciles (clases minoritarias). Es la solución nativa de Ultralytics para imbalance — equivalente en efecto a class weights, pero implementada como modificación de loss en lugar de modificar el gradient de cada sample.

**Advertencia:** Este experimento varía **dos** cosas respecto al baseline (capacidad + focal loss). Para atribuir la mejora correctamente, se analiza el mAP por clase en la sección de resultados: si las clases raras mejoran desproporcionadamente, se atribuye a focal loss; si todas mejoran uniformemente, se atribuye a mayor capacidad del modelo.

In [ ]:
set_seed(SEED)
model3 = YOLO("yolov8m.pt")

t0 = time.time()
results3 = model3.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    fl_gamma=1.5,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp3_yolov8m_focal",
    exist_ok=False,
    verbose=False,
)
t1 = time.time()
print(f"\nExp 3 completado en {(t1 - t0) / 60:.1f} minutos")

del model3
torch.cuda.empty_cache()

---
## 7. Tabla comparativa de experimentos

In [ ]:
def load_results(run_name: str) -> dict:
    csv_path = Path("runs/exp") / run_name / "results.csv"
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]  # quitar espacios en versiones viejas

    # Columnas de métricas (Ultralytics naming puede variar)
    map50_col     = next(c for c in df.columns if "mAP50(B)" in c or "mAP_0.5" in c)
    map5095_col   = next(c for c in df.columns if "mAP50-95(B)" in c or "mAP_0.5:0.95" in c)
    precision_col = next(c for c in df.columns if "precision(B)" in c.lower() or "precision" in c.lower())
    recall_col    = next(c for c in df.columns if "recall(B)"    in c.lower() or "recall"    in c.lower())

    best_idx = df[map5095_col].idxmax()
    best     = df.loc[best_idx]

    # Tiempo total en minutos
    train_time = None
    time_cols  = [c for c in df.columns if "time" in c.lower()]
    if time_cols:
        train_time = round(df[time_cols[0]].sum() / 60, 1)

    return {
        "run":          run_name,
        "best_epoch":   int(best_idx) + 1,
        "mAP@50":       round(float(best[map50_col]),   4),
        "mAP@50-95":    round(float(best[map5095_col]), 4),
        "Precision":    round(float(best[precision_col]), 4),
        "Recall":       round(float(best[recall_col]),   4),
        "tiempo_min":   train_time,
    }

exp_names = ["exp1_yolov8s_full", "exp2_yolov8s_frozen", "exp3_yolov8m_focal"]
rows = [load_results(n) for n in exp_names]

comparison_df = pd.DataFrame(rows).set_index("run")
comparison_df.index = [
    "Exp1: YOLOv8s full",
    "Exp2: YOLOv8s freeze",
    "Exp3: YOLOv8m focal",
]
print("=== Tabla comparativa de experimentos ===")
display(comparison_df)

best_run_name = exp_names[comparison_df["mAP@50-95"].values.argmax()]
print(f"\nMejor experimento (mAP@50-95): {best_run_name}")

### Análisis comparativo

> **Completar luego de ver los resultados.** Guía de interpretación:
> - **Exp1 vs Exp2:** Si Exp2 (freeze) supera a Exp1 (full), confirma la hipótesis de que con 1.200 imágenes el backbone COCO es ya suficiente y fine-tunear todo lo sobreescribe. Si Exp1 gana, las features específicas del dominio fútbol (camisetas, perspectiva broadcast) justifican el full fine-tune.
> - **Exp3 vs Exp1/2:** Si Exp3 gana principalmente en clases raras (Goal_Net, Corner), se atribuye a focal loss. Si gana en todas las clases, la capacidad del modelo m es el factor dominante.

---
## 8. Curvas de entrenamiento

In [ ]:
def plot_training_curves(run_name: str, ax_row: list, title_prefix: str):
    csv_path = Path("runs/exp") / run_name / "results.csv"
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    epochs = range(1, len(df) + 1)

    # Loss total train = sum de losses disponibles
    train_loss_cols = [c for c in df.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss_cols   = [c for c in df.columns if "val"   in c.lower() and "loss" in c.lower()]

    map50_col   = next(c for c in df.columns if "mAP50(B)" in c or "mAP_0.5" in c)
    map5095_col = next(c for c in df.columns if "mAP50-95(B)" in c or "mAP_0.5:0.95" in c)

    # Subplot 1: losses
    ax_loss = ax_row[0]
    if train_loss_cols:
        train_total = df[train_loss_cols].sum(axis=1)
        ax_loss.plot(epochs, train_total, label="Train loss", color="steelblue")
    if val_loss_cols:
        val_total = df[val_loss_cols].sum(axis=1)
        ax_loss.plot(epochs, val_total,   label="Val loss",   color="tomato", linestyle="--")
    ax_loss.set_title(f"{title_prefix}\nLoss (train + val)")
    ax_loss.set_xlabel("Época")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    # Subplot 2: mAP
    ax_map = ax_row[1]
    ax_map.plot(epochs, df[map50_col],   label="mAP@50",    color="green")
    ax_map.plot(epochs, df[map5095_col], label="mAP@50-95", color="darkgreen", linestyle="--")
    ax_map.set_title(f"{title_prefix}\nmAP (validación)")
    ax_map.set_xlabel("Época")
    ax_map.set_ylabel("mAP")
    ax_map.legend()
    ax_map.grid(True, alpha=0.3)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
labels = [
    ("exp1_yolov8s_full",   "Exp1: YOLOv8s Full"),
    ("exp2_yolov8s_frozen", "Exp2: YOLOv8s Freeze"),
    ("exp3_yolov8m_focal",  "Exp3: YOLOv8m Focal"),
]
for i, (run, label) in enumerate(labels):
    plot_training_curves(run, axes[i], label)

plt.suptitle("Curvas de entrenamiento — Comparativa de experimentos", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("dev/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardado: dev/training_curves.png")

### Análisis de overfitting / underfitting

> **Completar luego de ver las curvas.** Guía:
> - **Overfitting:** Train loss baja sostenidamente pero Val loss sube o se estanca. mAP val se amesetó. Early stopping activo.
> - **Underfitting:** Ambas losses altas al final. mAP val < 0.4 en dataset con objetos de tamaño medio. Considerar más épocas o mayor lr.
> - **Buen fit:** Train y val loss descienden juntas. mAP val sigue subiendo hasta early stopping.

---
## 9. Selección del mejor modelo

In [ ]:
best_run_name = exp_names[comparison_df["mAP@50-95"].values.argmax()]
BEST_WEIGHTS = Path("runs/exp") / best_run_name / "weights" / "best.pt"

assert BEST_WEIGHTS.exists(), f"No encontrado: {BEST_WEIGHTS}"
print(f"Mejor modelo: {best_run_name}")
print(f"Pesos:        {BEST_WEIGHTS}")
print(f"mAP@50-95 (val): {comparison_df['mAP@50-95'].max():.4f}")

---
## 10. Evaluación final en el conjunto de test

**Importante:** Se usa **test** (nunca visto), no validación. Las métricas de validación se usaron solo para seleccionar el modelo.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    plots=True,
    save_json=True,
    verbose=False,
)
print(f"\nmAP@50     (test): {test_metrics.box.map50:.4f}")
print(f"mAP@50-95  (test): {test_metrics.box.map:.4f}")

In [ ]:
# Métricas por clase
names = test_metrics.names  # dict {id: class_name}
n_classes = len(names)

p  = test_metrics.box.p   # array [n_classes]
r  = test_metrics.box.r
f1 = 2 * p * r / (p + r + 1e-9)
ap50   = test_metrics.box.ap50
ap5095 = test_metrics.box.maps

per_class_df = pd.DataFrame({
    "Clase":     [names[i] for i in range(n_classes)],
    "Precision": p.round(4),
    "Recall":    r.round(4),
    "F1":        f1.round(4),
    "mAP@50":    ap50.round(4),
    "mAP@50-95": ap5095.round(4),
}).set_index("Clase")

print("=== Métricas por clase (test set) ===")
display(per_class_df)

# Bar chart F1 por clase
fig, ax = plt.subplots(figsize=(10, 4))
per_class_df["F1"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(per_class_df["F1"].mean(), color="red", linestyle="--", label="Media F1")
ax.set_title("F1 por clase (test set)")
ax.set_xlabel("F1")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar matriz de confusión generada por Ultralytics
val_dir = Path("runs/exp") / best_run_name / "val_on_test"
# Ultralytics guarda plots en el directorio de validación
# Buscar confusion matrix
cm_paths = list(Path("runs/exp").glob(f"**/{best_run_name}*/confusion_matrix*.png"))
if not cm_paths:
    # Buscar en el directorio de val más reciente
    cm_paths = list(Path("runs/exp").glob("**/confusion_matrix*.png"))

if cm_paths:
    for cm_path in cm_paths[:2]:  # mostrar normal + normalizada si existen
        img = Image.open(cm_path)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(cm_path.name)
        plt.tight_layout()
        plt.show()
else:
    print("Matriz de confusión no encontrada en runs/. Verificar que plots=True en model.val().")

---
## 11. Análisis de errores

In [ ]:
TEST_IMAGES_DIR = Path("data/raw/test/images")
TEST_LABELS_DIR = Path("data/raw/test/labels")
IOU_THRESHOLD   = 0.5
CONF_THRESHOLD  = 0.25

def load_gt_boxes(label_path: Path, img_w: int, img_h: int) -> list:
    """Lee etiquetas YOLO y devuelve lista de (class_id, x1, y1, x2, y2) en píxeles."""
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:5])
        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        x2 = (xc + w / 2) * img_w
        y2 = (yc + h / 2) * img_h
        boxes.append((cls, x1, y1, x2, y2))
    return boxes

def iou(box_a, box_b) -> float:
    """IoU entre dos cajas (x1, y1, x2, y2)."""
    xa1, ya1, xa2, ya2 = box_a[1], box_a[2], box_a[3], box_a[4]
    xb1, yb1, xb2, yb2 = box_b[1], box_b[2], box_b[3], box_b[4]
    inter_x1 = max(xa1, xb1); inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2); inter_y2 = min(ya2, yb2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union  = area_a + area_b - inter + 1e-6
    return inter / union

# Correr predicciones en test
error_cases = []  # (img_path, gt_boxes, pred_boxes, error_type)

test_imgs = sorted(TEST_IMAGES_DIR.glob("*.jpg"))[:500]  # limitar a 500 para rapidez

for img_path in test_imgs:
    img = Image.open(img_path)
    w, h = img.size

    gt = load_gt_boxes(TEST_LABELS_DIR / (img_path.stem + ".txt"), w, h)
    if not gt:
        continue

    result = best_model.predict(source=str(img_path), conf=CONF_THRESHOLD,
                                 verbose=False, save=False)[0]
    preds = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            preds.append((cls_id, x1, y1, x2, y2, conf))

    # Detectar errores: falsos negativos o clasificación errónea
    for gt_box in gt:
        matched = False
        wrong_class = False
        for pred in preds:
            pred_box_fmt = (pred[0], pred[1], pred[2], pred[3], pred[4])
            if iou(gt_box, pred_box_fmt) > IOU_THRESHOLD:
                matched = True
                if pred[0] != gt_box[0]:
                    wrong_class = True
                break
        if wrong_class:
            error_cases.append((img_path, gt, preds, "clase_errónea"))
            break
        if not matched:
            error_cases.append((img_path, gt, preds, "falso_negativo"))
            break

print(f"Errores detectados en muestra de {len(test_imgs)} imágenes: {len(error_cases)}")
print(f"  Primeros 10: {[e[3] for e in error_cases[:10]]}")

In [ ]:
# Visualizar los 9 primeros casos de error
n_show = min(9, len(error_cases))
if n_show == 0:
    print("No se encontraron casos de error con los criterios actuales. Considerar bajar IOU_THRESHOLD.")
else:
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 6))
    axes = axes.flatten()

    for i, (img_path, gt, preds, error_type) in enumerate(error_cases[:n_show]):
        img_arr = np.array(Image.open(img_path))
        ax = axes[i]
        ax.imshow(img_arr)

        # Dibujar GT en verde
        for (cls_id, x1, y1, x2, y2) in gt:
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, CLASS_NAMES[cls_id], color="lime",
                    fontsize=7, backgroundcolor="black")

        # Dibujar predicciones en rojo
        for (cls_id, x1, y1, x2, y2, conf) in preds:
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       linewidth=2, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y2 + 10, f"{CLASS_NAMES[cls_id]} {conf:.2f}",
                    color="red", fontsize=7, backgroundcolor="white")

        ax.set_title(f"{img_path.name}\nError: {error_type}", fontsize=9)
        ax.axis("off")

    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Análisis de errores — GT (verde) vs Predicción (rojo)", fontsize=12)
    plt.tight_layout()
    plt.savefig("dev/error_analysis.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Guardado: dev/error_analysis.png")

### Causas posibles de error

> **Completar luego de ver las imágenes.** Causas típicas esperadas:
> - **TEAM 1 / TEAM 2 confundidos:** camisetas de colores similares, especialmente en condiciones de iluminación variable o cuando el jugador está parcialmente ocluido.
> - **Ball no detectada:** la pelota es el objeto más chico del dataset (bbox muy pequeña) y puede perderse en imágenes de alta densidad de jugadores o con motion blur de broadcast.
> - **Goal_Net / Corner falsos negativos:** clases con pocas instancias en entrenamiento; el modelo no tuvo suficiente exposición a estas clases.
> - **Oclusión:** jugadores superpuestos en jugadas de densidad alta; la predicción devuelve un solo bbox cuando en GT hay dos separadas.

---
## 12. Guardar modelo final

In [ ]:
DEST_MODEL = Path("dev/modelo.pth")
shutil.copy(BEST_WEIGHTS, DEST_MODEL)

size_mb = DEST_MODEL.stat().st_size / 1e6
print(f"Modelo guardado: {DEST_MODEL}")
print(f"Tamaño: {size_mb:.1f} MB")

# Verificar que carga correctamente
test_load = YOLO(str(DEST_MODEL))
print(f"Modelo cargado OK: {type(test_load.model).__name__}")
print()
print("NOTA: modelo.pth es un checkpoint Ultralytics (torch pickle con clase YOLO).")
print("Cargar con: YOLO('dev/modelo.pth')  — NO con torch.load() directo.")
del test_load

---
## 13. Verificación de requisitos

Checklist de requisitos del enunciado (Semana 3):

| Requisito | Sección | Estado |
|-----------|---------|--------|
| a) Modelo preentrenado + justificación + modificaciones + estrategia freeze | §3, §4-6 | ✅ |
| b) Loss, optimizador, LR, scheduler, épocas, batch, hardware, tiempo | §4-6 (celdas + markdown) | ✅ |
| c) ≥3 experimentos + tabla comparativa con métricas val + análisis | §4-7 | ✅ |
| d) Curvas loss+mAP train/val + comentario overfitting/underfitting | §8 | ✅ |
| e) Métricas test: P/R/F1 por clase + matriz de confusión | §10 | ✅ |
| f) Análisis de errores con ejemplos visuales | §11 | ✅ |
| g) Modelo guardado como `dev/modelo.pth` | §12 | ✅ |
| h) Notebook en `dev/02_model_training.ipynb` | — | ✅ |
| i) Reproducible end-to-end (seed, deterministic) | §1, §4-6 | ✅ |